## Investigating Indices
In this example, we're going to explore the sonification of the geomagnetic indices. 

We're going to approach this one index set at a time, hopefully introducing the context of each of these indices and designing sonifications that could be useful for space weather monitoring. The principle of this example is to spend more time thinking about **sonification design choices** rather than thinking about how PLUM and SoRBET operate.

 That's not to say they won't play a major role - quite the opposite, they underpin everything we do here, we'll just be **thinking about clever ways to use these building blocks to make something more intricate in this example**

In [ ]:
# Standard imports
import numpy as np
import matplotlib.pyplot as plt

#For later audio processing, we import Audio, and install + import pydub
from IPython.display import Audio, display
%pip --quiet install pydub
from pydub import AudioSegment

#Point to relevant subroutines and data folders
import sys
sys.path.insert(0, '../subroutines/')
sys.path.insert(0, '../Data/')

# Import the ManGOE object sonification routine
from PLUM import PitchSonification, PitchEventSonification, CutoffSonification, PanSonification
from ManGOE import ManGOE_Object, ManGOE_Event

## 1) The Kp Index

In [ ]:
time_data, Kp = np.loadtxt('../Data/patricks_day_2015_Kp_indices.csv', delimiter=',', skiprows=1, unpack=True)

We will start with an event sonification in this case, and because we're not doing anything fancy with the sound at this stage.

Here, I supply 9 notes on the E1 major scale (as there are 9 classical Kp bins) and create a PitchEventSonification from PLUM.

In [ ]:
Base_Notes = [["E1","F1","G1","A1","B1","C2","D2","E2","F2","G2"]]

length = 45

Kp_lims = (0,9)

soni_base = PitchEventSonification(Base_Notes,length,time_data,Kp,pitch_lims = Kp_lims,downsample=1)

First_soni = soni_base.notebook_display()

Now we have a basis to work with, what we're going to do is **augment** this basis piece by adding supplementary chanels to it. Turns out, you can make some good things when you take simple things and combine them.

The first technique we're going to use is **Thirds and Fifths** - where we make the sound richer at points in the sonification we'd like people to notice. In the case of the Kp index, we would like people to start noticing during storm times, and ESPECIALLY during strong storms.

So, we will apply thirds whenever the Kp index is greater than 5 (in the NOAA space weather framework, category G1 or more), and apply fifths when the storm is severe, meaning Kp>7 (and NOAA category G3 or more)

So we first determine our thirds and fifths of the notes we had for the base sonification, and make new data which represents the Kp>5 and Kp>7 categories using pythons ability to mask arrays. Then, we sonify them!

In [ ]:
Thirds = [["G1","G#1","A#1","C2","D2","D#2","F2","G2","G#2","A#2"]]
Fifths = [["A#1","B1","C#2","D#2","F2","F#2","G#2","A#2","B2","C#3"]]

G12 = Kp[Kp > 5]
time_G12 = time_data[Kp > 5]

G34 = Kp[Kp > 7]
time_G34 = time_data[Kp > 7]

time_lims = (time_data[0],time_data[-1]*1.01)

soni_G12 = PitchEventSonification(Thirds,length,time_G12,G12,pitch_lims = Kp_lims,time_lims=time_lims,downsample=1)

if len(G34)!=0:
    soni_G34 = PitchEventSonification(Fifths,length,time_G34,G34,pitch_lims = Kp_lims,time_lims=time_lims,downsample=1)
else:
    print('there are no severe storms and so sonifications of that part of the data cannot be completed')



Now, to combine these, we use the same strategy as the PLUM tutorial - save the sonification as .wav files, use pydub's `AudioSegment` to extract their audio segments, use the overlay call to combine them and save the result.

In [ ]:
soni_base.save('Kps.wav')
soni_G12.save('Thirds.wav')
soni_G34.save('Fifths.wav')

# Load your two files
track1 = AudioSegment.from_wav("Kps.wav")
track2 = AudioSegment.from_wav("Thirds.wav")
track3 = AudioSegment.from_wav("Fifths.wav")

# Overlay them (mix together, starting at the same time)
combined = track1.overlay(track2).overlay(track3)

# Export
combined.export("Kp_Experience.wav", format="wav")

# If you've saved to file
print('Below is the combined sound file of both sonifications to make a combined experience!')
display(Audio("Kp_Experience.wav"))

We can also do this using object sonifications as well, and the best way to do that is to **control the volume**. Instead of providing a mask to the Kp index, we instead make arrays that are precisely 1 when the Kp index is above the value we want and 0 otherwise - **meaning there will be no sound when the condition isn't true**. 

These arrays can be made using python's `np.where` function, and then we use ManGOE to generate the third and the fifths with the volume mapped to the volume controls. This ensures the thirds and the fifths only "switch on" at the right time.

In [ ]:
warning_volume = np.where(Kp>5,1,0) 
doom_volume = np.where(Kp>7,1,0) 

base_note = [["E1"]]

warning_note = [["G1"]]

doom_note = [["A#1"]]

warning_maps = {
    'volume':warning_volume,
    'pitch_shift':Kp,
    'time_evo':time_data
}

doom_maps  = {
    'volume':doom_volume,
    'pitch_shift':Kp,
    'time_evo':time_data
}

soni_Kp = PitchSonification(base_note,length,time_data,Kp)
soni_warning = ManGOE_Object(warning_note,length,warning_maps)
soni_doom = ManGOE_Object(warning_note,length,warning_maps)

We then combine the three sonifications together as we did for the events case.

In [ ]:
soni_Kp.save('Kps.wav')
soni_warning.save('warning.wav')
soni_doom.save('doom.wav')

# Load your two files
track1 = AudioSegment.from_wav("Kps.wav")
track2 = AudioSegment.from_wav("warning.wav")-6
track3 = AudioSegment.from_wav("doom.wav")-12

# Overlay them (mix together, starting at the same time)
combined = track1.overlay(track2).overlay(track3)

# Export
combined.export("Kp_Experience.wav", format="wav")

# If you've saved to file
print('Below is the combined sound file of both sonifications to make a combined experience!')
display(Audio("Kp_Experience.wav"))